# Training Embedding Models for Dense Retrieval in RAG Systems

Retrieval-Augmented Generation (RAG) systems combine information retrieval with language generation. A *dense retrieval* component uses neural **embedding models** to encode texts (queries and documents) into high-dimensional vectors, such that relevant pairs have high similarity. In this lecture, we cover how these embedding models are trained for RAG, discuss popular model architectures (BAAI BGE, Microsoft E5, Jina Embeddings, ColBERT, etc.), their training objectives (contrastive losses, triplet losses), and evaluation metrics for retrieval. We also highlight the latest state-of-the-art models as of late 2024–mid 2025, especially those excelling in multilingual retrieval.

**Table of Contents**:

1. [Dense Retrieval with Embeddings](#dense-retrieval-overview)  
2. [Key Embedding Models for Dense Retrieval](#embedding-models)  
   - 2.1 [BAAI **BGE** (General Embedding)](#bge)  
   - 2.2 [Microsoft **E5**](#e5)  
   - 2.3 [Jina **Embeddings**](#jina)  
   - 2.4 [**ColBERT** (Contextualized Late Interaction)](#colbert)  
3. [Training Objectives: Contrastive and Triplet Losses](#training-losses)  
4. [Evaluation Metrics for Retrieval](#evaluation-metrics)  
5. [Recent State-of-the-Art Models (2024–2025)](#sota-models)


## Introduction and Context 

Retrieval-Augmented Generation (RAG) combines large language models with an external knowledge retrieval component to produce more factual and contextually relevant outputs. A central piece of RAG is the **retriever** – a system that fetches relevant documents or passages based on a query. Traditionally, information retrieval relied on **sparse retrieval**, using methods like TF-IDF or BM25 that represent text as sparse, high-dimensional vectors of token counts. In sparse retrieval, most dimensions are zero, indicating absence of specific words; for example, a document about “machine learning” has high values for those terms and zeros for unrelated words ([What is the difference between sparse and dense retrieval?](https://milvus.io/ai-quick-reference/what-is-the-difference-between-sparse-and-dense-retrieval#:~:text=The%20key%20difference%20lies%20in,For)). **Dense retrieval**, in contrast, uses neural networks to encode text into continuous low-dimensional vectors (embeddings) where every dimension has a value. These dense embeddings capture semantic meaning, enabling similarity comparison even without shared keywords. In the context of RAG, dense retrieval allows a user’s query to find relevant passages by *meaning*, which can then be supplied to the generation model as additional context.

**Sparse vs. Dense Retrieval:** Sparse methods (e.g. BM25) excel at precise keyword matching – they are fast and effective when exact terms matter, but they struggle with synonyms or rephrased queries. Dense methods use transformers (like BERT) to group related concepts in vector space; e.g. “movie” and “film” end up close together, so a query *“sci-fi movies”* may retrieve a passage about “science fiction films” even if it doesn’t contain the exact words. Dense retrieval thus handles synonyms and semantic context well. However, dense models require substantial training data and compute, and can be biased by their training domain. In practice, many systems use a **hybrid approach**: combining sparse and dense retrieval to leverage the strengths of both. For example, a hybrid retriever might use BM25 to ensure precise term overlap and a dense model to catch semantic matches.

**Retrieval-Augmented Generation Pipeline:** In a typical RAG system, documents are encoded into embeddings and stored in a vector index (often a vector database). At query time, the query is encoded into the same embedding space and the nearest neighbor search finds the top-$k$ relevant passages ([Retrieval Augmented Generation](https://www.ibm.com/architectures/patterns/genai-rag#:~:text=2,it%20easier%20to%20find%20information)). Those passages are then provided (often concatenated) as additional input to the language model, which generates the final answer or text. The performance of RAG heavily depends on the quality of the retriever – if the retrieved passages are irrelevant, the generator may produce incorrect results ([Dense Retrieval in RAG: Must-Read Papers Before Implementation | by BettyHCZhang | Medium](https://medium.com/@287961061/dense-retrieval-in-rag-must-read-papers-before-implementation-2a91b4e72298#:~:text=LLMs%20such%20as%20GPT,first%20step%20in%20the%20procedure)). This is why **dense retrievers** have become crucial: they often provide better semantic recall of relevant info than sparse methods, especially for open-ended or natural language queries.

In recent years, dense retrieval models have advanced rapidly. A milestone was the **Dense Passage Retrieval (DPR)** system by Karpukhin et al. (2020), which demonstrated that a dual-encoder BERT model trained on question–passage pairs can outperform a strong BM25 baseline by 9–19% in top-20 passage recall on open QA benchmarks ([[2004.04906] Dense Passage Retrieval for Open-Domain Question Answering](https://arxiv.org/abs/2004.04906#:~:text=%3E%20Abstract%3AOpen,domain%20QA)). DPR’s success kicked off a wave of research into training better embedding models for retrieval. We now have a variety of models – both from academia and industry – that push the state of the art in dense retrieval. This lecture will explore how such **embedding models for dense retrieval** are trained and evaluated, focusing on several prominent examples: BAAI’s **BGE** models, Microsoft’s **E5** models, **Jina** embeddings, and the **ColBERT** late-interaction model, among others. We’ll discuss their architectures, training methodologies, benchmarks, and use cases, with an eye to how they fit into RAG systems.

Before diving into specific models, we will briefly overview the key benchmarks and metrics used to evaluate retrieval models.

## Key Retrieval Benchmarks and Evaluation Metrics

When developing dense retrieval models, researchers evaluate them on standard **information retrieval benchmarks** to measure how well they find relevant documents. Some important benchmarks and datasets include:

- **MS MARCO (Microsoft MAchine Reading COmprehension)**: A large-scale dataset originally released in 2016, containing real Bing queries and passages with human-labeled answers ([[2004.04906] Dense Passage Retrieval for Open-Domain Question Answering](https://arxiv.org/abs/2004.04906#:~:text=%3E%20Abstract%3AOpen,domain%20QA)). The popular MS MARCO *passage ranking* task has ~1 million training queries and is used to supervise dense retrievers. Models are typically evaluated on a dev set (or the TREC Deep Learning tracks which stem from MS MARCO) using metrics like MRR@10 (Mean Reciprocal Rank at 10) and Recall@100. MS MARCO is a key dataset for training; many dense models (DPR, ColBERT, etc.) are fine-tuned on it.

- **Natural Questions (NQ)** and **TriviaQA**: Open-domain question answering benchmarks where a system must retrieve Wikipedia passages containing the answer. These test a retriever’s ability to find factual answers. A common metric is Recall@k: e.g., the percentage of questions for which the correct answer is contained in the top-5 or top-20 retrieved passages. Dense retrievers like DPR were originally shown to dramatically improve top-20 recall on NQ over BM25, enabling higher QA accuracy.

- **BEIR (Benchmarking IR)**: A **heterogeneous zero-shot retrieval benchmark** that aggregates 18 datasets across diverse tasks and domains (e.g. web search, biomedical, finance, tweet retrieval, Q&A) ([[2104.08663] BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models](https://arxiv.org/abs/2104.08663#:~:text=address%20this%2C%20and%20to%20facilitate,In%20contrast%2C%20dense%20and)). Models are evaluated on BEIR *without further training* (“zero-shot”) to assess generalization. The main metric is nDCG@10 (Normalized Discounted Cumulative Gain at rank 10), which accounts for graded relevance. BEIR revealed that in 2021, **BM25 was a very tough baseline** – many dense models trained only on one domain fell short on BEIR’s diverse tasks, unless combined with heavy re-rankers. This highlighted the need for more robust, general-purpose embedding models.

- **MTEB (Massive Text Embedding Benchmark)**: A recently introduced benchmark that evaluates embedding models across a wide range of tasks – including retrieval, clustering, classification, and semantic textual similarity – in multiple languages. It has 56 datasets and reports an overall score ([A Guide to Open-Source Embedding Models](https://www.bentoml.com/blog/a-guide-to-open-source-embedding-models#:~:text=NV,spot%20on%20the%20same%20leaderboard)) ([A Guide to Open-Source Embedding Models](https://www.bentoml.com/blog/a-guide-to-open-source-embedding-models#:~:text=wide%20variety%20of%20tasks%2C%20ranking,spot%20on%20the%20same%20leaderboard)). MTEB’s retrieval subset overlaps with BEIR. MTEB is useful for assessing an embedding model’s *versatility* beyond just pure retrieval. For example, in 2023 some models (like Nvidia’s NV-Embed-v2 or BAAI’s BGE) topped this benchmark with high overall scores.

- **Specialized Benchmarks**: There are also benchmarks focusing on multilingual retrieval (e.g. **MIRACL** for many languages, or **MKQA** for multilingual QA) and domain-specific ones (e.g. clinical or legal document retrieval). For instance, BGE-M3’s creators tested it on MIRACL (multilingual information retrieval across 16 languages) and MKQA (multi-lingual QA), where it achieved new state-of-the-art results ([GitHub - FlagOpen/FlagEmbedding: Retrieval and Retrieval-augmented LLMs](https://github.com/FlagOpen/FlagEmbedding#:~:text=%2A%201%2F30%2F2024%3A%20Release%20BGE,Technical%20Report)).

**Evaluation Metrics:** The common metrics in retrieval include: **Recall@K** (did we retrieve the relevant doc in the top K results?), **MRR** (which rewards higher rank of the first relevant result), and **nDCG** (which accounts for multiple relevance levels and positions). In open-domain QA settings, a downstream metric is Exact Match accuracy of the answer, but the retriever’s quality is usually measured by passage recall. In general, a good dense retriever for RAG should have high recall (finding most of the relevant info in its top candidates) and reasonably good precision so that the top results are actually useful to the generator.

Having set the stage, we now turn to **major dense retrieval models**. We’ll examine how each is architected and trained, what benchmarks it shines on, and how it’s applied, especially in RAG contexts. We start with the **BAAI General Embedding (BGE)** model family, which exemplifies a modern multi-purpose embedding approach.

## 1. Dense Retrieval with Embeddings <a name="dense-retrieval-overview"></a>

Dense retrieval uses **bi-encoders** (dual neural networks) to embed queries and documents into a shared vector space. A query’s embedding is matched to the nearest document embeddings via cosine or dot-product similarity. This approach overcomes lexical mismatch issues of classical sparse methods like BM25 by capturing semantic similarity ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=Text%20embeddings%20are%20low,easily%20consumable%20by%20downstream%20applications)) ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=framework%20to%20enhance%20the%20sequence,of%20unlimited%20quantity%2C%20they%20are)). In a RAG pipeline, the dense retriever finds candidate passages for a given question/query, and then an LLM generates the answer using those passages as context.

Traditionally, each text is represented by a single *d*-dimensional vector (e.g., *d* = 768). The embedding model is usually a Transformer encoder (e.g. BERT) that produces a vector (often using the [CLS] token’s output or an average pooling over token outputs). At query time, the vector dot-product (or cosine) serves as a relevance score for retrieval. Dense embedding models must be trained so that relevant query–document pairs have higher similarity than irrelevant pairs.

**Training data:** Many embedding models leverage large-scale *weakly supervised* data: pairs of texts likely to be relevant (for example, question–answer pairs from forums, similar sentences, or even synthetic pairs). Models can also be fine-tuned on labeled QA or IR datasets to further improve relevance.

**Contrastive learning** is the dominant training paradigm: the model is taught to **contrast** a positive pair against negatives. Below we discuss popular models and the loss functions they use to learn such embeddings.

## 2. Key Embedding Models for Dense Retrieval <a name="embedding-models"></a>

### 2.1 BAAI **BGE** (General Embedding) <a name="bge"></a>

**Overview:** BGE, from the Beijing Academy of AI, is a family of *general-purpose* embedding models aimed at supporting all kinds of retrieval and ranking tasks ([Recent advances in text embedding: A Comprehensive Review of Top-Performing Methods on the MTEB Benchmark](https://arxiv.org/html/2406.01607v1#:~:text=across%20various%20application%20settings%20such,g)). The BGE models use a BERT-based architecture where the final layer’s [CLS] token is trained to be the embedding for the input text. They perform *instruction-based fine-tuning*: during training, a task description (e.g. *“Find relevant passages for the query:”*) is prepended to queries to help the model handle different tasks. This helps BGE serve diverse applications (search, QA, recommendation, etc.) with a single unified model.

**Training data:** BGE introduced a *C-Pack* dataset for Chinese (and similar large datasets for other languages) to pre-train the embedding model. For English, unsupervised corpora like Wikipedia, CC-News, Reddit, etc., were used for pre-training, and labeled datasets (NLI, MSMARCO, Quora duplicates, etc.) for fine-tuning. Pre-training focuses on making the encoder produce generally useful embeddings, while fine-tuning refines it on specific tasks (with instructions guiding each task).

**Training objective:** BGE is trained with a **contrastive loss**. In pre-training, it uses *in-batch negatives* and extremely large batch sizes (up to 19k) to sample many negative examples. In fine-tuning, it also adds one **hard negative** (a challenging irrelevant example mined from the corpus) for each positive pair. The loss function is the standard InfoNCE *contrastive loss*, which encourages the query’s embedding $q$ to be more similar to its positive document $d^+$ than to any negative $d^-$ in the batch. Formally, for a batch $B$ of $N$ query–positive pairs $(q_i, d_i^+)$ and a set of negatives $\{d_j^-\}$ (e.g. other pairs’ positives acting as negatives), the loss for query $i$ is: 

$$
L_{\text{contrastive}}^{(i)} = -\log \frac{\exp(\text{sim}(q_i,\; d_i^+)/\tau)}{\sum_{j=1}^{N} \exp(\text{sim}(q_i,\; d_j^-)/\tau)} \,,
$$

where $\text{sim}(q,d)$ is a similarity score (often cosine similarity or dot product) and $\tau$ is a temperature hyperparameter. Intuitively, $L_{\text{contrastive}}$ is minimized when $q_i$ is much closer to its true match $d_i^+$ than to any other text in the batch.

**PyTorch-style snippet (InfoNCE Loss with in-batch negatives):**
```python
import torch, torch.nn.functional as F
# q_emb, d_emb: [batch_size, dim] normalized embeddings for queries and docs
similarity_matrix = torch.matmul(q_emb, d_emb.T)        # shape [N, N], dot-product similarities
similarity_matrix /= tau                                # scale by temperature
labels = torch.arange(similarity_matrix.size(0))        # label i should match doc i
loss = F.cross_entropy(similarity_matrix, labels)       # cross-entropy over N-way softmax
```

In the above code, each query *i* treats the *i*-th document as positive and all others in the batch as negatives, implementing the InfoNCE loss.

**Usage and Performance:** BGE models are highly versatile. A single model can be used for dense retrieval, multi-vector retrieval, and even generate sparse (lexical) representations ([GitHub - FlagOpen/FlagEmbedding: Retrieval and Retrieval-augmented LLMs](https://github.com/FlagOpen/FlagEmbedding#:~:text=%2A%201%2F30%2F2024%3A%20Release%20BGE,Technical%20Report)). BGE’s multilingual model (BGE-M3) supports 100+ languages and achieved state-of-the-art on multilingual retrieval benchmarks like MIRACL (multilingual retrieval). Early BGE versions ranked 1st on the MTEB benchmark (Massive Text Embedding Benchmark) for both English and Chinese subsets. This demonstrates that with large-scale contrastive training and instruction tuning, BGE produces strong embeddings across diverse tasks and languages.

### 2.2 Microsoft **E5** (Embeddings from Bidirectional Encoder Representations) <a name="e5"></a>

**Overview:** E5 is a family of embedding models from Microsoft that set a new standard for general-purpose text embeddings ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=In%20this%20work%2C%20we%20learn,Scientific%20papers%2C%20and%20perform%20aggressive)). Introduced in late 2022, E5 models are trained with *weak supervision* on a massive dataset of text pairs, and were the **first to surpass BM25** (a strong traditional IR baseline) on the BEIR benchmark under zero-shot (no task-specific training) conditions. When fine-tuned on labeled data, E5 models also achieved state-of-the-art results on MTEB, outperforming even other embedding models that had 40× more parameters.

**Architecture:** E5 uses a Transformer encoder (based on BERT or RoBERTa) to produce a single 768d or 1024d embedding for each input. Notably, E5 employs **prefix tokens** to distinguish query vs. document input types: they prepend `"query: "` or `"passage: "` to the text before encoding ([Brief Review — Text Embeddings by Weakly-Supervised Contrastive Pre-training | by Sik-Ho Tsang | Medium](https://sh-tsang.medium.com/brief-review-text-embeddings-by-weakly-supervised-contrastive-pre-training-c799c319bcfa#:~:text=,to%20be%20stable%2C%20and%20outperforms)). This breaks the symmetry and helps when the model is used for asymmetric tasks like search (where “query” and “passage” distributions differ). E5 often uses **average pooling** of the Transformer outputs (instead of [CLS]) to form the embedding, and applies a scaling factor (temperature $\tau$) to the cosine similarity during training. For example, the E5-base model uses $\tau=0.01$, effectively magnifying similarity differences.

**Training data (CCPairs):** E5 is pretrained on *Colossal Clean Pairs (CCPairs)* , a web-scale collection of ∼270 million unlabeled text pairs mined from diverse sources (Reddit Q&A, StackExchange Q&A, Wikipedia title–passage, news titles–articles, etc.). These pairs are obtained through heuristic filtering and a novel consistency filtering to ensure quality ([Microsoft’s E5 Text Embedding Model Tops the MTEB Benchmark With 40x Fewer Parameters | Synced](https://syncedreview.com/2022/12/13/microsofts-e5-text-embedding-model-tops-the-mteb-benchmark-with-40x-fewer-parameters/#:~:text=In%20their%20first%20step%2C%20the,text%20pairs%20for%20contrastive%20pretraining)). The scale and diversity of CCPairs helped E5 learn very robust semantic representations. 

**Loss function:** Like BGE, E5 relies on a **contrastive InfoNCE loss** with in-batch negatives. During pre-training, for each pair $(q, p)$, other pairs’ passages in the batch serve as negatives. The scoring function is $s_\theta(q,p) = \cos(E(q), E(p)) / \tau$ as described above. The InfoNCE loss is the same form as given for BGE. In practice, E5’s training used large batch sizes (e.g. 2048) to ensure many negatives ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=Another%20critical%20issue%20for%20contrastive,the%20batch%20size%20is%20sufficiently)) and found this outperformed more complex negative mining strategies. 

After this unsupervised contrastive pre-training, E5 undergoes a **second-stage fine-tuning** on a small set of high-quality supervised data . The authors combined **NLI** (natural language inference) data, **MS MARCO** passage ranking, and **Natural Questions** datasets for fine-tuning. Each of these contributes something: NLI provides sentence similarity judgments (entailment vs contradiction) which help semantic textual similarity tasks; MS MARCO and NQ provide actual IR supervision for QA tasks. During fine-tuning, they also perform **hard negative mining** and **knowledge distillation** from a cross-encoder teacher (a more expensive but accurate re-ranker). The fine-tuning loss is a *weighted sum* of the contrastive loss on hard labels and a KL-divergence loss to match the teacher’s score distribution. In formula form, for student scores producing softmax $p_{\text{stu}}$ and teacher soft labels $p_{\text{ce}}$, the fine-tune loss was: 

$$ 
L_{\text{fine-tune}} = \alpha \, L_{\text{contrastive}} + (1-\alpha)\,D_{\mathrm{KL}}(p_{\text{ce}} \parallel p_{\text{stu}})\,,
$$ 

balancing between the hard positives/negs and the teacher’s relevance signal.

**PyTorch-style snippet (Contrastive loss for E5 pre-training):**
```python
# Assume we have N query embeddings and N positive passage embeddings
sim = torch.matmul(query_emb, passage_emb.T)         # [N,N] similarity scores
sim /= tau                                           # apply temperature scaling (e.g., tau=0.01)
labels = torch.arange(N, dtype=torch.long)
contrastive_loss = F.cross_entropy(sim, labels)
```
*(Fine-tuning adds additional terms like distillation loss; omitted here for brevity.)*

**Results:** E5’s contrastive training proved extremely effective. Without using any labeled data, **E5-base** surpassed BM25 on BEIR (15 dataset average), becoming the first embedding model to do so in zero-shot retrieval. Scaling up to larger models further improved zero-shot performance (E5-large beat E5-base). With supervised fine-tuning, E5-large achieved an nDCG@10 of 48.7 on BEIR, outperforming prior SOTA GTR-large. On the massive MTEB benchmark, E5 models not only *substantially* outperformed other models of similar size, but even matched or beat models far larger. For instance, a fine-tuned E5-base outperformed GTR-XXL (a T5-3B based model) on many tasks. This efficiency—strong performance with relatively few parameters—made E5 very attractive in practice. 

**Takeaway:** E5 demonstrates the power of **weakly supervised contrastive learning** at scale. By training on hundreds of millions of diverse text pairs, the model learns embeddings that generalize to many tasks (retrieval, clustering, classification) without further tuning ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=well%20to%20a%20wide%20range,best%20results%20on%20the%20MTEB)). Fine-tuning with a careful blend of tasks and distillation then further boosts performance on specific benchmarks. E5’s recipe (large-scale contrastive pre-training + multi-task fine-tune) has since influenced many subsequent embedding models.

### 2.3 **Jina Embeddings** (v2, v3) <a name="jina"></a>

**Overview:** Jina AI has developed open-source embedding models optimized for **multilingual and multitask** performance. Notably, **jina-embeddings-v3** (released Sept 2024) is a 570M parameter model that achieved state-of-the-art on multilingual data and long-context retrieval tasks ([Jina Embeddings V3: Multilingual Embeddings With Task LoRA](https://arxiv.org/pdf/2409.10173#:~:text=We%20introduce%20jina,out%02performs%20the%20latest%20proprietary%20embeddings)). It supports input lengths up to 8192 tokens, making it suitable for long documents. Jina’s models are designed to be competitive with or outperform proprietary embeddings (OpenAI, Cohere) while being open-source.

**Architecture:** Jina Embeddings v3 uses a Transformer encoder backbone (570M params, significantly larger than BERT-base but smaller than 7B LLMs). It incorporates **Low-Rank Adaptation (LoRA)** adapters for different tasks. Instead of having one static embedding model for all purposes, Jina v3 includes multiple task-specific adapter modules that can be applied on top of the base encoder: for example, separate adapters for *query-document retrieval*, *clustering*, *classification*, *textual similarity matching*, etc. This design allows the base model to share general linguistic knowledge, while each adapter fine-tunes the representation for a particular type of task. Importantly, all these tasks use a common vector space, and Jina v3 introduces *Matryoshka Representation Learning* to make the embeddings **dimension-flexible**: the embedding vector can be truncated to smaller sizes (e.g., from 1024-d to 256-d or 32-d) with minimal loss in performance. This is useful for efficiency, as one can choose a smaller dimensional index if needed without retraining the model.

**Training approach:** Jina’s embedding models are trained with a **multi-task learning** approach. During training, batches of data are drawn from various tasks and languages, and the model (with relevant adapter) optimizes a loss specific to that task. For retrieval tasks, Jina uses a **contrastive InfoNCE loss** similar to E5 and BGE (often with in-batch negatives, possibly plus hard negatives). For example, jina-v2 was trained with a pairwise NCE loss over positive and in-batch negative pairs. Jina v3 extends this with additional negatives and a *triplet loss* component. Specifically, Jina v3 mentions using an $L_{\text{triplet}}$ in addition to NCE, to account for an extra hard negative in the batch ([Jina Embeddings V3: Multilingual Embeddings With Task LoRA](https://arxiv.org/pdf/2409.10173#:~:text=match%20at%20L478%20loss%20Ltriplet,Ltriplet%28B%29%20%3A%3D%20Er%E2%88%BCB)). The triplet loss encourages a margin between the positive and hardest negative. For classification tasks (e.g. assigning class labels or clustering), Jina uses specialized losses such as **CoSent loss** (Consistency Sentence Ranking loss) or separation losses that encourage embeddings of the same class to cluster and different classes to be far apart. Each task’s data is fed in separately so that, for instance, a batch contains only retrieval data (processed with a contrastive loss), or only classification data (with a class-separation loss). The overall training objective is a weighted sum of all these task-specific losses.

In summary, Jina v3’s training objective $L_{\text{total}}$ might look like a combination of multiple terms:
$$
L_{\text{total}} = \lambda_{\text{ret}} L_{\text{InfoNCE}} + \lambda_{\text{triplet}} L_{\text{triplet}} + \lambda_{\text{sts}} L_{\text{CoSent}} + \cdots,
$$
with different $\lambda$ weights for retrieval, semantic similarity, classification, etc., losses. This multi-task cocktail forces the model to produce embeddings that are simultaneously good at retrieving relevant texts, grouping similar meanings, and separating different classes.

**Triplet loss example:** A typical *triplet margin loss* is defined over a query $q$, a positive document $d^+$, and a negative document $d^-$. It aims to ensure $q$ is closer to $d^+$ than to $d^-$ by a margin. One common formulation is:

$$
L_{\text{triplet}} = \max\{0, \; m + \text{sim}(q,\;d^-) - \text{sim}(q,\;d^+)\} \,,
$$

where $m$ is the margin (a small constant like 0.2). The loss is zero if the similarity of the positive pair exceeds the negative pair by at least $m$, otherwise it grows linearly. Jina v3’s use of triplet loss helps it incorporate additional negatives beyond the in-batch negatives.

**PyTorch-style snippet (Triplet loss):**
```python
import torch.nn.functional as F
# Assume embeddings for query, positive doc, negative doc (batch of vectors)
pos_score = F.cosine_similarity(query_emb, pos_emb)   # similarity between q and positive
neg_score = F.cosine_similarity(query_emb, neg_emb)   # similarity between q and negative
margin = 0.2
# Triplet margin ranking loss: maximize pos_score - neg_score by at least margin
loss = F.relu(margin + neg_score - pos_score).mean()
```

In practice, frameworks also provide `nn.TripletMarginLoss` for such objectives. Jina’s training likely alternates between such losses depending on task. 

**Multilingual and long-text support:** A major focus of Jina’s models is multilingual capability. Jina Embeddings v2 and v3 are trained on data covering many languages (v3 cites training on 104 languages) and domains. This wide coverage, plus the ability to handle long input lengths (up to 8192 tokens), makes them suitable for tasks like retrieving long documents or cross-lingual search. Despite the relatively large context, Jina v3 still maintains high efficiency compared to using full large language models for embedding, since 570M is much smaller than typical 7B LLMs.

**Performance:** Jina Embeddings v3 achieved excellent results on MTEB. It **outperforms OpenAI and Cohere’s latest embeddings on English tasks**, and also **surpasses the best open multilingual model (e.g. multilingual-E5-large)** on *all* multilingual tasks ([Jina Embeddings v3: A Frontier Multilingual Embedding Model](https://jina.ai/news/jina-embeddings-v3-a-frontier-multilingual-embedding-model/#:~:text=Evaluation%20on%20the%20MTEB%20benchmark,org%20Saba%20Sturua)). This means Jina’s embeddings are not only competitive with closed-source offerings, but actually set a new state-of-the-art in many benchmarks. For example, with a default 1024-d embedding, Jina v3 topped clustering, retrieval, and classification metrics across languages. A remarkable feature is that these embeddings can be truncated to as low as 32 dimensions with negligible performance loss, thanks to the Matryoshka training technique ([Jina Embeddings V3: Multilingual Embeddings With Task LoRA](https://arxiv.org/pdf/2409.10173#:~:text=from%20OpenAI%20and%20Cohere%20on,instruct)). This gives practitioners flexibility to trade off some accuracy for a huge gain in index size and speed, which is valuable in industrial deployments.

In summary, **Jina’s contribution** is showing that a carefully crafted *multitask, multilingual training regime* on a moderately sized model can yield **universal embeddings** that excel in many scenarios. By integrating ideas like LoRA adapters per task, mixing contrastive and triplet losses, and handling long contexts, Jina v3 represents a bridge between academic advances (like instruction tuning, multi-negative training) and real-world needs (multilingual support, efficiency). It validates that one model can serve as a *one-stop solution* for query/document embedding in a multilingual RAG system, simplifying the system architecture.

### 2.4 **ColBERT** (Contextualized Late Interaction) <a name="colbert"></a>

So far, we focused on *single-vector* embedding models. **ColBERT** (Khattab & Zaharia, 2020) takes a different approach: it’s a *multi-vector* retrieval model that retains multiple embeddings per document to allow fine-grained term matching ([[2402.03216] BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings Through Self-Knowledge Distillation](https://ar5iv.org/pdf/2402.03216.pdf#:~:text=common%20form%20of%20embedding,2020)). ColBERT stands for *Contextualized Late Interaction over BERT*. It’s designed to combine the speed of embedding retrieval with some of the term-specific matching accuracy of traditional IR.

**Architecture:** In ColBERT, queries and documents are encoded by BERT-based encoders **without reducing them to a single vector**. Instead, each token (word-piece) in the query and each token in the document get their own embedding vector (after a linear projection and $\ell_2$ normalization). For example, a query of *m* tokens might be represented as $\{q_1,\dots,q_m\}$ and a document of *n* tokens as $\{d_1,\dots,d_n\}$, where each $q_i$ and $d_j$ are 128-dimensional vectors in the original ColBERT. To compute a relevance score, ColBERT uses a **Late Interaction** mechanism called *MaxSim*: for each query token embedding $q_i$, it finds the *maximum* similarity with any document token embedding, $\max_{j}\text{sim}(q_i, d_j)$. Then it sums these maxima over all query tokens:
$$
s(q, d) \;=\; \sum_{i=1}^m \;\max_{1 \le j \le n} \;\text{sim}(q_i,\; d_j)\,. 
$$
Typically $\text{sim}(u,v)$ is dot product or cosine similarity. This scoring function means: each query term tries to find a document term that best matches it, and contributes that score, and finally we aggregate all query terms’ scores. Intuitively, the query is satisfied if for each of its terms, the document has some very similar term. This captures exact- or synonym-matching signals much better than a single global embedding can (since a single vector might “average out” the individual terms).

Because documents are represented by many vectors, ColBERT retrieval is heavier than single-vector methods. However, thanks to efficient vector-indexing (the document token embeddings can be pre-indexed) and the fact that computing MaxSim is fast with dot products on accelerators, ColBERT can still scale to large corpora (especially ColBERTv2 which introduced optimizations). It supports *multi-vector retrieval*: effectively each document is a set of vectors.

**Training objective:** ColBERT is trained in a supervised fashion on labeled query-document pairs (e.g., from MS MARCO) to maximize the score of relevant pairs and minimize the score of irrelevant ones. The original ColBERT used a **pairwise cross-entropy loss**: for a given query, with one positive document and one negative document, it applies a softmax over their scores and maximizes the probability of the positive. In other words, given $s(q,d^+)$ and $s(q,d^-)$, the model is optimized to assign a higher score to $d^+$:
$$
L = -\log \frac{\exp(s(q,\;d^+))}{\exp(s(q,\;d^+)) + \exp(s(q,\;d^-))} \,. 
$$
This is equivalent to a binary classification where the model must pick which of the two documents is relevant to the query. More generally, if multiple negatives are present, a softmax can be taken over the positive and all negatives, akin to the InfoNCE loss. ColBERT’s training essentially treats the positive as class 1 and all negatives as class 0 for each query.

**PyTorch-style snippet (Pairwise softmax loss for ColBERT):**
```python
# Assume we have a batch of query embeddings, each with a positive and a negative document
pos_scores = model(query_inputs, pos_doc_inputs)   # [batch] scores
neg_scores = model(query_inputs, neg_doc_inputs)   # [batch] scores
# Stack scores so that for each query we have [score_pos, score_neg]
scores = torch.stack([pos_scores, neg_scores], dim=1)   # shape [batch, 2]
labels = torch.zeros(scores.size(0), dtype=torch.long)  # label 0 = positive doc index
loss = F.cross_entropy(scores, labels)
```
Here `model(query, doc)` would compute the ColBERT score (using MaxSim under the hood). The loss pushes `score_pos` higher than `score_neg` for each query.

**In practice**, large batches and multiple negatives (in-batch or mined) are used. ColBERT can also leverage *hard negatives* from BM25 during training (provide particularly challenging non-relevant docs). The pairwise softmax loss is a form of contrastive loss; it is actually the same as InfoNCE with one positive and one negative.

**Discussion:** ColBERT’s late interaction allows it to handle cases like a query word that must appear in the document. Single-vector models sometimes struggle with such exact constraints because they embed the *whole sentence* meaning. ColBERT, however, can pick up that if the query has the word “Java”, the document needs a very similar token (like “Java” itself or a synonym) to satisfy that term’s MaxSim. This approach yields **higher recall and precision** on benchmarks like MSMARCO passage ranking compared to single-vector bi-encoders at the time. It became a popular choice for passage search when accuracy was paramount.

The trade-off is that ColBERT’s index is larger (many vectors per document) and query-time computation is heavier (summing over query terms). Subsequent improvements (ColBERTv2, etc.) have reduced memory and increased speed, narrowing the gap. ColBERT and related *late interaction* models represent an interesting middle ground between plain dual encoders and full cross-attention models: they allow some cross-interaction (at the token level via MaxSim) but remain indexable and scalable like dense retrieval.

**Takeaway:** ColBERT shows that modifying the representation (multiple vectors) and similarity function (MaxSim) can significantly improve retrieval quality, at some cost in efficiency. In the context of RAG, ColBERT-style models might be chosen when the highest precision is required and resources allow, or for domains where word-level matching is crucial. Many modern systems still use simpler single-vector embeddings for speed, but understanding ColBERT’s approach is valuable, as it has inspired newer multi-vector retrievers and re-ranking strategies.

## 3. Training Objectives: Contrastive vs. Triplet Loss <a name="training-losses"></a>

In training dense retrievers, **contrastive loss** and **triplet loss** are two common objectives. We have seen these in the context of specific models above, but here we summarize their forms and intuition:

- **Contrastive (InfoNCE) Loss:** This is a classification-based loss that treats the training batch as a classification problem: for each query $q_i$, the model should “identify” its true document $d_i^+$ among a set of candidates. It is also known as *InfoNCE* (info noise-contrastive estimation) or *softmax* loss. Given a batch $\{(q_i, d_i^+)\}_{i=1}^N$, we typically use all other $d_j^+$ ($j \neq i$) as negatives for $q_i$. The loss was given earlier, but in simpler terms, for each $i$:
  $$L_i = -\log \frac{\exp(\text{sim}(q_i, d_i^+))}{\sum_{j=1}^{N} \exp(\text{sim}(q_i, d_j^+))}\,. $$
  This encourages $\text{sim}(q_i, d_i^+)$ to be higher than $\text{sim}(q_i, d_j^+)$ for any $j \neq i$. It is a *global* objective (comparing a positive against all negatives simultaneously). Contrastive loss benefits from *many negatives*, which is why methods like in-batch negatives or large batches or memory banks (MoCo) are used ([Text Embeddings by Weakly-Supervised Contrastive Pre-training](https://arxiv.org/html/2212.03533v2#:~:text=Another%20critical%20issue%20for%20contrastive,the%20batch%20size%20is%20sufficiently)). The temperature $\tau$ (often implicitly applied) scales the softness of the softmax; a low $\tau$ (e.g. 0.05) makes the softmax concentrate more on the highest-scoring item, acting almost like a hinge.

- **Triplet Loss:** This is a *margin ranking* loss applied to triplets $(q, d^+, d^-)$. It doesn’t normalize over many negatives at once, instead it looks at one (or a few) negatives at a time and enforces a margin. A common form:
  $$L_{\text{triplet}}(q, d^+, d^-) = \max\{0,\; m + \text{dist}(q,d^+) - \text{dist}(q,d^-)\}\,,$$ 
  where $\text{dist}(\cdot,\cdot)$ is a distance (often $1 - \cos$ similarity) and $m$ is the margin. This ensures the distance to the positive is at least $m$ smaller than the distance to the negative (equivalently $\text{sim}(q,d^+)$ is $m$ larger than $\text{sim}(q,d^-)$). If the positive is already much closer than the negative, the loss is 0 (no need to adjust). Triplet loss thus focuses on *relative ordering* of a positive and a negative for the same query. It’s a simpler objective and often requires careful mining of a strong negative to be effective (otherwise easy negatives yield zero loss). In practice, people often mine *hard negatives* (e.g. using BM25 or the model itself iteratively) and use a triplet loss or a variant (sometimes with multiple negatives per anchor in a summed form).

**When to use which?** Contrastive InfoNCE loss with many negatives tends to perform better when batch size/GPU memory allows, as it uses more information (each query is contrasted with many negatives, not just one). It provides a stronger learning signal and is smooth (the softmax provides gradient to push down on *all* negatives, weighted by their scores). Triplet loss is easier to implement if you only can sample one or a few negatives per query – it doesn’t require a huge softmax computation. Some older sentence embedding works (e.g., Sentence-BERT) used triplet loss with one hard negative due to batch size limitations, whereas newer ones use in-batch negatives with cross-entropy. 

In multi-task scenarios, sometimes both are used: e.g., use InfoNCE for one part of training and triplet for another, as we saw with Jina’s approach. Also, **contrastive loss** is a subset of a broader family of *ranking losses*. One can also use a binary cross-entropy (logistic regression) approach: $\log\sigma(s(q,d^+) - s(q,d^-))$, which is mathematically very close to the softmax formulation.

**Bottom line:** Both losses aim to make $s(q,d^+) > s(q,d^-)$ for positives vs negatives. Contrastive (softmax) does this by pushing on the entire list of negatives at once; triplet does it by local pair comparison with a margin. When implemented correctly, both can yield good results; however, InfoNCE with large-batch negatives is almost the default in recent research because of its superior empirical performance.

## 4. Evaluation Metrics for Retrieval <a name="evaluation-metrics"></a>

After training an embedding model, we need to evaluate how well it retrieves relevant documents. Several standard **evaluation metrics** are used in information retrieval (IR). Here we expand on three common metrics with their formulas and usage context:

- **Recall@K:** Recall at rank *K* measures the fraction of relevant documents that are retrieved in the top *K* results for each query. If a query $q$ has a set of relevant documents $R(q)$ in the corpus, and the retrieval system returns a ranked list $\text{ranked}_K(q)$ of the top $K$ results, then 
  $$\text{Recall@}K = \frac{1}{|Q|}\sum_{q \in Q} \frac{|\,R(q) \cap \text{ranked}_K(q)\,|}{|R(q)|}\,,$$
  where the outer sum averages over all queries in the set $Q$. In words, for each query we compute the proportion of its relevant items found in the top $K$, and then average. In cases where each query is assumed to have at least one relevant document, sometimes a simplified definition is used: the percentage of queries for which *at least one* relevant document is in the top $K$. (This is effectively assuming $|R(q)|=1$ for each or just measuring success rate.) **Usage:** Recall@K is very important when there are multiple relevant documents or when we care about finding *any* relevant doc in the top results. For example, in open-domain QA, there may be several passages containing the answer; we want our retriever to get at least one of them in the top $K$ (so that the reader can find the answer). Recall@10 or Recall@100 are common metrics in retrieval benchmarks like MS MARCO and BEIR when evaluating the first-stage retriever. A high Recall@K means the retriever is catching most of the possible relevant info in its top-K list (good for coverage). Recall does not consider the rank positions among the top K or any non-relevant in between, it’s a set-based measure for the cutoff.

- **MRR (Mean Reciprocal Rank):** This metric focuses on the rank position of the *first relevant result*. The reciprocal rank for a single query is $1/r$ where $r$ is the rank of the highest-ranked relevant document for that query. If no relevant document is retrieved, the reciprocal rank is typically taken as 0. The **mean reciprocal rank** is then the average of these reciprocals over all queries:
  $$ \text{MRR} = \frac{1}{|Q|}\sum_{q \in Q} \frac{1}{\text{rank}_q} \,, $$
  where $\text{rank}_q$ is the rank position of a relevant doc for query $q$ (the topmost one). For example, if for one query the first relevant answer is at rank 1, its reciprocal rank is 1; if for another query the first relevant is at rank 5, reciprocal rank is 0.2; and if none in top K (infinite rank), we treat it as 0. Then average these values. **Usage:** MRR is commonly used in scenarios like QA or web search where typically there’s one primary correct answer/document per query and we care a lot about getting that answer as high as possible. MS MARCO’s official metric for the passage ranking task was MRR@10 (MRR calculated with cutoff at 10, meaning if no relevant in top 10, it’s 0) ([ColBERT: Efficient and Effective Passage Search via Contextualized ...](https://training.continuumlabs.ai/knowledge/vector-databases/colbert-efficient-and-effective-passage-search-via-contextualized-late-interaction-over-bert#:~:text=,higher%20scores%20to%20positive)). MRR heavily rewards getting the answer at rank 1 vs rank 2, etc., due to the reciprocal. It’s a good metric when each query has a single relevant item (or you only care about the first relevant item retrieved). In cases with multiple relevant documents, MRR only accounts for the first one and ignores if there were more later. So it’s not used when recall of all relevant is important.

- **nDCG (Normalized Discounted Cumulative Gain):** nDCG is a popular metric, especially when relevance judgments are **graded** (e.g., not just binary relevant/non-relevant, but say on a 3-point scale). It also accounts for the rank of *all* retrieved relevant documents, with diminishing returns for lower ranks. The calculation has a few steps:
  1. Compute DCG@K (Discounted Cumulative Gain at cutoff K) for each query. Given the ranked list of results, each result at rank $i$ has a relevance grade $rel_i$ (e.g., 0 for non-relevant, 1 for relevant, or higher for highly relevant). The DCG formula is: 
     $$ \text{DCG@}K = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)} \,. $$
     This formulation (using $2^{rel}-1$) is one common version which assumes relevance grades on an exponential scale of gain. If relevances are binary (0/1), this simplifies to summing $1/\log_2(i+1)$ for each relevant at rank $i$. The denominator $\log_2(i+1)$ means that higher-ranked documents contribute more gain (since the denominator grows with rank, thus discounting lower ranks). The first position has $\log_2(2)=1$ in denominator, so full weight; rank 2 has $\log_2(3)≈1.585$, so relevant at rank 2 contributes about 0.63 of what it would at rank 1, and so on. This models the idea that a relevant result is less useful if it’s seen later by the user.
  2. Compute IDCG@K (Ideal DCG) for each query, which is the DCG@K for that query if all the most relevant documents were ranked at the top in ideal order. Essentially, sort the query’s documents by relevance and calculate DCG up to K. This represents the *maximum possible gain* one could achieve for that query.
  3. **nDCG@K** is the normalized DCG: $\text{nDCG@}K = \frac{\text{DCG@}K}{\text{IDCG@}K}$. This yields a number between 0 and 1 for each query (with 1 meaning perfect ranking). Finally, usually we average nDCG@K over all queries in the set.

**Usage:** nDCG (often at K=10 or 20 for retrieval tasks) is widely used in academic IR evaluations (TREC competitions, Web search benchmarks) because it handles multiple relevant documents and graded relevance. For example, in web search, some results are highly relevant, others somewhat relevant; nDCG accounts for that by the gain values. BEIR benchmark uses nDCG@10 as the primary metric to compare models ([Brief Review — Text Embeddings by Weakly-Supervised Contrastive Pre-training | by Sik-Ho Tsang | Medium](https://sh-tsang.medium.com/brief-review-text-embeddings-by-weakly-supervised-contrastive-pre-training-c799c319bcfa#:~:text=Supervised%20Fine)). Even for binary relevance, nDCG@10 is a fine measure of ranking quality, as it rewards getting more relevant documents and getting them earlier in the ranking. Compared to Recall@K, nDCG@K also cares about the order of those relevant documents (i.e., having a relevant at rank 1 and none until rank 10 is better for nDCG than having them all appear rank 6-10). It’s a balanced metric for scenarios where each query can have many relevant docs of varying importance.

**Other metrics:** There are others like Precision@K (the fraction of retrieved top K that are relevant), which is used sometimes (e.g., Precision@1 = accuracy of the top result). In retrieval, precision is less emphasized at the first-stage retrieval because usually one cares about getting as many relevant as possible (recall) for a second-stage re-ranker or reader. But for final results to users, precision or nDCG are more indicative of quality. Another metric is **MRR@K** which means MRR but only considering ranking up to K (similar to how MS MARCO reported MRR@10).

In summary, to evaluate dense retrievers:
- We use **Recall@K** to ensure the model retrieves enough relevant candidates (important in multi-answer or for feeding into downstream systems).
- We use **MRR** (or Hits@1) when there’s usually one right answer and we care about ranking it first.
- We use **nDCG@K** (or sometimes Recall+Precision or MAP) when there are multiple relevant documents with different importance, to evaluate the overall ranking quality.

Typically, benchmarking suites (like BEIR, MTEB) will report nDCG@10 as a primary metric ([Brief Review — Text Embeddings by Weakly-Supervised Contrastive Pre-training | by Sik-Ho Tsang | Medium](https://sh-tsang.medium.com/brief-review-text-embeddings-by-weakly-supervised-contrastive-pre-training-c799c319bcfa#:~:text=Supervised%20Fine)), and sometimes Recall@100 for retrievers. MRR is common in QA or community QA settings (like evaluating how well the top answer was found).

*Example:* If our dense retriever is part of a QA system, we might check Recall@100 on a large corpus (can it retrieve at least one correct answer passage in top 100?). If it’s high, the reader model has a good chance. If comparing two retrievers on a search task with graded relevance, we’d compare their nDCG@10 — a model with higher nDCG@10 is delivering more useful results in the top 10.

## 5. Recent State-of-the-Art Embedding Models (late 2024 – mid 2025) <a name="sota-models"></a>

The field of dense retrieval is evolving rapidly. From late 2024 into 2025, we’ve seen many **LLM-based embedding models** that push performance to new heights, especially in multilingual settings. Many top models now leverage large language model backbones (7B+ parameters) with clever fine-tuning or distillation to create powerful bi-encoders. Here are some notable state-of-the-art models and trends:

- **Alibaba **GTE** (General Text Embedding, Qwen-7B-instruct)**: Alibaba’s GTE is built on their Qwen-7B large language model, adapted for use as a dual encoder. It introduces **bidirectional attention** (making the decoder operate like an encoder) and instruction tuning for embedding tasks. Trained on a large multilingual corpus, GTE has been a **top performer on MTEB** for both English and Chinese benchmarks. Its key contributions are leveraging a powerful base model (Qwen-7B), multi-domain instruction tuning, and making a decoder-only model behave like an encoder for embeddings. GTE’s success showed that an LLM can be repurposed into an excellent embedding model with minimal architectural changes.

- **BAAI **BGE M3**:** BGE’s latest iteration “M3” (Multilingual, Multi-Function, Multi-Granularity) in 2024 extends the BGE approach with **self-knowledge distillation** among dense, sparse, and multi-vector retrieval modes ([GitHub - FlagOpen/FlagEmbedding: Retrieval and Retrieval-augmented LLMs](https://github.com/FlagOpen/FlagEmbedding#:~:text=%2A%201%2F30%2F2024%3A%20Release%20BGE,Technical%20Report)). It supports 100+ languages and inputs up to 8192 tokens. By combining dense retrieval training with lexical and ColBERT-style retrieval signals (teacher scores from different methods), BGE M3 learns a truly versatile embedding. It achieved new SOTA on multilingual retrieval benchmarks like MIRACL (21 languages retrieval) and cross-lingual QA (MKQA). BGE-M3 demonstrates that a single model can unify different retrieval paradigms. Its multilingual capability also ranked among the best on Chinese and English MTEB in 2023. This model is especially relevant for systems needing a *unified* solution across languages and retrieval styles.

- **Jina **Embeddings v3**:** As discussed, jina-embeddings-v3 (570M param) came out in 2024 and **outperforms even the latest proprietary models from OpenAI and Cohere on many English tasks**, and beats the strong multilingual-e5-large model on all tested multilingual tasks ([Jina Embeddings v3: A Frontier Multilingual Embedding Model](https://jina.ai/news/jina-embeddings-v3-a-frontier-multilingual-embedding-model/#:~:text=Evaluation%20on%20the%20MTEB%20benchmark,org%20Saba%20Sturua)). With an embedding dimension of 1024 (flexible down to 32), it strikes an excellent balance between model size and performance. It incorporates *task-specific LoRA adapters* for specialization and was trained on a broad range of tasks (retrieval, clustering, etc.). On the **MTEB leaderboard**, Jina v3 is one of the top models, proving that carefully fine-tuned mid-sized models can challenge much larger LLM-based embeddders. For practitioners, Jina v3 offers an open-source, license-friendly alternative to API models, with strong multilingual prowess (supporting 100+ languages).

- **Microsoft **E5-mistral-7B-instruct**:** This is an extension of the E5 family using a 7B *Mistral* LLM as the backbone (Mistral is a 7B open model, an improved Llama2). Microsoft combined the pretraining approach of E5 with the capacity of a 7B model, and instruction-tuned it (“instruct” indicates using natural language instructions in training). The result is a very powerful embedding model that leverages the knowledge of a larger language model. E5-mistral-7B demonstrated one approach to using LLMs: generate synthetic training data via the LLM and then train a bi-encoder. In fact, Microsoft used LLMs to create synthetic queries for passages, a bit akin to the “FRet” method below, to supplement training data. The E5-mistral model, when evaluated, achieved among the highest scores on English retrieval tasks and greatly improved cross-lingual performance (since Mistral is trained on multilingual data). It underscores a trend: **starting from a strong foundation model and fine-tuning it for retrieval yields excellent results** with relatively little training needed (the LLM already has a lot of knowledge).

- **Echo & LLM2Vec:** These are experimental approaches showing that **decoder-only LLMs can produce good embeddings without any synthetic data, by enabling bidirectional context** . Normally, GPT-style models are not ideal for embeddings because they are unidirectional. But techniques like adding a prefix and fine-tuning (LLM2Vec) or slight architecture tweaks can let a Llama2 or Mistral model generate embeddings. For example, LLM2Vec took a 7B decoder model and fine-tuned it with a small encoder on top, to get embeddings effectively from the LLM’s knowledge. These models (LLM2Vec, and another called Echo) achieved surprisingly strong performance (they appear in MTEB rankings not far behind the top). They didn’t rely on LLM-synthesized data, instead they use the LLM’s internal representation ability. This line of research suggests even lighter ways to get universal embeddings from LLMs.

**Common trends:** Almost all top models in 2025 use some form of **instruction tuning** – they feed natural language descriptions of tasks or contexts during training ([Recent advances in text embedding: A Comprehensive Review of Top-Performing Methods on the MTEB Benchmark](https://arxiv.org/html/2406.01607v1#:~:text=that%20good%20universal%20text%20embeddings,7B%20model%20is%20the)). This helps the model better align with what we want (e.g., “Given a query, find relevant passages…” as an instruction). Also, **LoRA adapters** are widely used for fine-tuning these large models efficiently. And importantly, using an existing **LLM as a starting point** (whether to generate training data, or as the model itself) is a recurrent theme. This is because LLMs already encode a lot of semantic knowledge; with some coaxing, they can be turned into excellent retrievers with relatively little additional data. 

On the multilingual front, models like GTE, BGE-M3, Jina v3, and multilingual-E5 have ensured that non-English tasks are not left behind. In fact, many of these models report strong results on multilingual subsets of MTEB (which includes 112 languages). The gap between English and other languages is closing, thanks to large multilingual training data and translation techniques.

**Implications for RAG:** For building a RAG system today, one has a rich set of choices. If working with many languages, models like **BGE-M3 or Jina v3** provide ready-to-use multilingual embeddings. If focusing on English and seeking top quality, a model like **SFR-Embedding** or **E5-mistral** might give the best results (if one can handle the model size). For a balanced option, **Gecko** shows you can get high quality with a smaller model by investing in synthetic data generation. As open models, these can be run on your own infrastructure, enabling privacy and customization.

Finally, it’s worth noting that as of mid-2025, the research community is pushing towards **universal embedding models** that can do “all tasks” (retrieval, rerank, question similarity, etc.). The Massive Text Embedding Benchmark (MTEB) ([Top embedding models for RAG | Modal Blog](https://modal.com/blog/embedding-models-article#:~:text=The%20MTEB%20leaderboard%3A%20A%20benchmark,for%20embedding%20models)) has been a driving force, encouraging models to be evaluated on a wide array of tasks. This has led to innovation in making embeddings more *task-aware* yet still general (via instructions). We can expect future models to continue this trend, perhaps using even larger backbones (like 13B or 70B LLMs distilled down) or more sophisticated training pipelines. 

For now, a practitioner should keep an eye on the MTEB leaderboard (on HuggingFace Spaces) to see the *current best* models and consider factors like model size, multilingual needs, and license. The good news is that the gap between proprietary and open models has largely closed in this area – in fact, as we saw, open models like Jina v3 and SFR are leading, which is great for academia and industry alike, enabling powerful RAG systems without relying on closed APIs.

